In [0]:
# Create Product Dimension
dim_products = (spark.table("workspace.dev_silver_layer.events_cleaned")
                .select("product_id", "category_code", "brand")
                .distinct()) # Get unique product attributes

dim_products.write.mode("overwrite").saveAsTable("workspace.dev_gold_layer.dim_products")

In [0]:
from pyspark.sql.functions import count, sum

# Create Sales Fact
fact_sales = (spark.table("workspace.dev_silver_layer.events_cleaned")
              .filter("event_type = 'purchase'")
              .groupBy("product_id") # Group only by the key
              .agg(
                  count("event_type").alias("total_purchases"),
                  sum("price").alias("total_revenue")
              ))

fact_sales.write.mode("overwrite").saveAsTable("workspace.dev_gold_layer.fact_sales")

In [0]:
%sql
-- This view simplifies the Star Schema for BI users while enforcing security
CREATE OR REPLACE VIEW workspace.dev_gold_layer.v_final_sales_report AS
SELECT 
    p.brand,                   -- Row Filtering will apply here automatically
    p.category_code,
    f.total_revenue,
    f.total_purchases
FROM workspace.dev_gold_layer.fact_sales f
JOIN workspace.dev_gold_layer.dim_products p 
  ON f.product_id = p.product_id; -- Optimized by Liquid Clustering

In [0]:
spark.sql("""
  ALTER TABLE workspace.dev_gold_layer.fact_sales 
  CLUSTER BY (product_id)
""") #

In [0]:
%sql
-- Consolidates small files into large, efficient 1GB files
OPTIMIZE workspace.dev_gold_layer.fact_sales;
OPTIMIZE workspace.dev_gold_layer.dim_products;

-- Cleans up old file versions to save storage costs (keep 7 days)
VACUUM workspace.dev_gold_layer.fact_sales RETAIN 168 HOURS;